In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')



dane = pd.read_csv("adult.csv")
dane = dane.replace("?",np.nan).dropna()
y = dane['income'].map({'<=50K': 0,'>50K': 1})
X = pd.get_dummies(dane.drop('income', axis=1), drop_first=True).astype(float)


X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

X_train_sub = X_train.sample(n=5000, random_state=42)
y_train_sub = y_train.loc[X_train_sub.index]


X_train_base = sm.add_constant(pd.DataFrame(index=X_train_sub.index))
base_model = sm.Logit(y_train_sub, X_train_base).fit(disp=0, method='bfgs', maxiter=15)
best_aic = base_model.aic

selected_features = []
remaining_features = list(X_train_sub.columns)


while remaining_features:
    aic_with_cands = []
    for cand in remaining_features:
        features_to_test = selected_features + [cand]
        X_train_temp = sm.add_constant(X_train_sub[features_to_test])
        try:
            model = sm.Logit(y_train_sub, X_train_temp).fit(disp=0, maxiter=15, method='bfgs')
            aic_with_cands.append((model.aic, cand))
        except:
            continue
        
    aic_with_cands.sort()
    best_cand_aic, best_cand = aic_with_cands[0]
    if best_cand_aic < best_aic:
        selected_features.append(best_cand)
        remaining_features.remove(best_cand)
        best_aic = best_cand_aic
        print(f"Dodano zmienną: {best_cand} (AIC: {best_aic:.2f})")
    else:
        print("Brak poprawy kryterium AIC. Koniec procedury.")
        break
    
print(f"Ostateczne zmienne w modelu: {selected_features}")
X_train_final = sm.add_constant(X_train[selected_features])
final_model = sm.Logit(y_train, X_train_final).fit(disp=0)

print(final_model.summary())

X_test_final = sm.add_constant(X_test[selected_features])
y_pred_prob = final_model.predict(X_test_final)
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred))
        

Dodano zmienną: marital.status_Married-civ-spouse (AIC: 4509.33)
Dodano zmienną: education.num (AIC: 3921.10)
Dodano zmienną: capital.gain (AIC: 3861.27)
Dodano zmienną: relationship_Own-child (AIC: 3800.98)
Dodano zmienną: occupation_Exec-managerial (AIC: 3746.18)
Dodano zmienną: occupation_Other-service (AIC: 3717.57)
Dodano zmienną: occupation_Machine-op-inspct (AIC: 3702.68)
Dodano zmienną: education_Prof-school (AIC: 3690.55)
Dodano zmienną: occupation_Handlers-cleaners (AIC: 3678.79)
Dodano zmienną: education_Masters (AIC: 3668.46)
Dodano zmienną: education_Doctorate (AIC: 3658.53)
Dodano zmienną: education_Bachelors (AIC: 3642.12)
Dodano zmienną: capital.loss (AIC: 3615.80)
Dodano zmienną: education_11th (AIC: 3605.57)
Dodano zmienną: occupation_Prof-specialty (AIC: 3597.19)
Dodano zmienną: education_7th-8th (AIC: 3590.73)
Dodano zmienną: sex_Male (AIC: 3584.42)
Dodano zmienną: occupation_Farming-fishing (AIC: 3579.04)
Dodano zmienną: occupation_Transport-moving (AIC: 3575.91)
D